# Compound Events and Independence

A single event asks: "did this one thing happen?" A compound event asks: "did *this* happen AND *that* happen?" or "did *this* happen OR *that* happen?"

These are the same logical operators you use in code every day — `and`, `or` — applied to sets of outcomes. Python's set operations (`&`, `|`) handle them directly.

The concept of **independence** then answers a deeper question: does knowing one event occurred change the odds of the other? If not, the events are independent, and their probabilities multiply. This has a famous, widely-misunderstood consequence: the **gambler's fallacy**.

**Prerequisites:** Notebook 08 — Probability Basics.

**This is part 2 of 3.** Part 3 covers conditional probability and Bayes' theorem.

---

## Quick Recap: The Setup

From notebook 08: the sample space is the full set of outcomes; an event is any subset; probability is `len(event) / len(sample_space)` for uniform distributions.

We will use a standard 52-card deck throughout this notebook — a sample space large enough to make all the cases interesting.

In [ ]:
# Build the deck once; reuse throughout
suits  = ['H', 'D', 'C', 'S']
values = ['2','3','4','5','6','7','8','9','10','J','Q','K','A']
deck   = {(v, s) for v in values for s in suits}

def P(event, space=deck):
    """Probability of event within sample space (uniform distribution)."""
    return len(event) / len(space)

red_cards  = {card for card in deck if card[1] in ('H', 'D')}
face_cards = {card for card in deck if card[0] in ('J', 'Q', 'K')}
aces       = {card for card in deck if card[0] == 'A'}

print(f"Deck size:   {len(deck)}")
print(f"Red cards:   {len(red_cards)}")
print(f"Face cards:  {len(face_cards)}")
print(f"Aces:        {len(aces)}")

---

## Union (OR): At Least One Occurs

The **union** of two events A and B is the set of outcomes where at least one of A or B occurs. In Python: `A | B`.

The probability formula:

$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

The subtraction removes double-counting. If you just add `P(A) + P(B)`, any outcome that belongs to *both* events gets counted twice. Subtracting `P(A ∩ B)` — the intersection — corrects that.

Think of it like a SQL `UNION` with deduplication: you want every row that matches condition A or condition B, but you only want each row once.

In [ ]:
# P(Red OR Face card)
red_or_face  = red_cards | face_cards    # set union
red_and_face = red_cards & face_cards    # overlap — we'll need this

print(f"P(Red)             = {len(red_cards)}/{len(deck)} = {P(red_cards):.4f}")
print(f"P(Face)            = {len(face_cards)}/{len(deck)} = {P(face_cards):.4f}")
print(f"P(Red AND Face)    = {len(red_and_face)}/{len(deck)} = {P(red_and_face):.4f}  <- the overlap")
print()
print(f"P(Red OR Face) via set union:    {P(red_or_face):.4f}")
print(f"P(Red OR Face) via formula:      {P(red_cards) + P(face_cards) - P(red_and_face):.4f}")
print()

# Why does the overlap exist? 2 red suits × 3 face cards each = 6 red face cards
print(f"Red face cards: {sorted(red_and_face)}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Venn diagram: Red vs Face cards
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

circle_red  = plt.Circle((3.5, 3), 2.3, color='salmon',     alpha=0.4, label='Red cards (26)')
circle_face = plt.Circle((6.5, 3), 2.3, color='steelblue',  alpha=0.4, label='Face cards (12)')
ax.add_patch(circle_red)
ax.add_patch(circle_face)

ax.text(2.5, 3,   'Red only\n20',        ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(5.0, 3,   'Both\n6',             ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(7.5, 3,   'Face only\n6',        ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(5.0, 5.5, 'Red OR Face = 32 cards (= 20 + 6 + 6)',
        ha='center', fontsize=11, style='italic')

ax.set_title('P(A ∪ B) = P(A) + P(B) − P(A ∩ B)  |  26/52 + 12/52 − 6/52 = 32/52', fontsize=11)
plt.tight_layout()
plt.show()

---

## Intersection (AND): Both Occur

The **intersection** of A and B is the set of outcomes where both occur. In Python: `A & B`.

How you calculate `P(A ∩ B)` depends on whether the events are **independent**:

| Relationship | Formula |
|---|---|
| Independent | $P(A \cap B) = P(A) \times P(B)$ |
| Dependent | $P(A \cap B) = P(A) \times P(B \mid A)$ |

Two events are **independent** if knowing one occurred gives you no information about the other. The simplest test: does `P(A ∩ B) == P(A) * P(B)`?

In [ ]:
# Are Red and Face independent?
p_red        = P(red_cards)
p_face       = P(face_cards)
p_red_x_face = p_red * p_face      # what P(R and F) would be IF independent
p_red_and_face = P(red_and_face)   # what P(R and F) actually is

print(f"P(Red) × P(Face)       = {p_red:.4f} × {p_face:.4f} = {p_red_x_face:.4f}")
print(f"P(Red AND Face) actual = {p_red_and_face:.4f}")
print(f"Independent? {abs(p_red_x_face - p_red_and_face) < 1e-10}")
print()
print("Why? Face cards split evenly: 6 red face cards out of 12 total.")
print("The colour of a card and whether it's a face card carry no information about each other.")

In [ ]:
# Two fair coins — a classic independent setup
coin_space = {(a, b) for a in ('H', 'T') for b in ('H', 'T')}

first_heads  = {outcome for outcome in coin_space if outcome[0] == 'H'}
second_heads = {outcome for outcome in coin_space if outcome[1] == 'H'}
both_heads   = first_heads & second_heads

p_first  = P(first_heads,  coin_space)
p_second = P(second_heads, coin_space)
p_both   = P(both_heads,   coin_space)

print(f"Sample space: {coin_space}")
print(f"P(1st = H)         = {p_first}")
print(f"P(2nd = H)         = {p_second}")
print(f"P(both H) actual   = {p_both}")
print(f"P(1st H) × P(2nd H)= {p_first * p_second}")
print(f"Independent: {abs(p_both - p_first * p_second) < 1e-10}")

---

## The Complement: NOT

The complement of event A is everything in the sample space that is *not* A:

$$P(A^c) = 1 - P(A)$$

This is often the fastest route to calculating a probability. Instead of counting all the ways something *can* happen, count the ways it *cannot* and subtract from 1.

In [ ]:
# P(at least one head in two coin flips)
# Direct: count {HH, HT, TH} = 3/4
# Complement: 1 - P(no heads) = 1 - P(TT) = 1 - 1/4 = 3/4

coin_space   = {(a, b) for a in ('H', 'T') for b in ('H', 'T')}
no_heads     = {outcome for outcome in coin_space if 'H' not in outcome}
at_least_one = coin_space - no_heads   # set difference = complement

print(f"P(no heads)          = {P(no_heads, coin_space)}")
print(f"P(at least one head) = {P(at_least_one, coin_space)}")
print(f"Via complement:      = 1 - {P(no_heads, coin_space)} = {1 - P(no_heads, coin_space)}")
print()

# When the complement is much easier:
# P(at least one ace in 5 draws WITH replacement)
p_no_ace_single = 1 - 4/52
p_no_ace_five   = p_no_ace_single ** 5   # independent draws
p_at_least_one  = 1 - p_no_ace_five

print(f"P(at least one ace in 5 draws with replacement):")
print(f"  P(not ace) = {p_no_ace_single:.4f}")
print(f"  P(no ace in 5) = {p_no_ace_five:.4f}")
print(f"  P(at least one ace) = 1 - {p_no_ace_five:.4f} = {p_at_least_one:.4f}")

---

## The Gambler's Fallacy

Independence has an important consequence that trips people up constantly: **past outcomes of an independent process have no effect on future outcomes**.

If you flip a fair coin and get heads 10 times in a row, the probability of heads on the next flip is still exactly 0.5. The coin has no memory. Each flip is independent.

The **gambler's fallacy** is the belief that tails is now "due" — that the universe will rebalance. It won't.

Where the confusion comes from: over a *large* number of flips, you *do* expect roughly 50% heads (Law of Large Numbers). But that convergence happens by accumulating more data in the future, not by the future compensating for the past. A streak of 10 heads becomes a rounding error once you've accumulated 10,000 flips — it doesn't get cancelled out by a compensating streak of tails.

In [ ]:
import numpy as np

np.random.seed(42)

# After 10 heads in a row, what is the probability of heads on the 11th flip?
# Simulate 100,000 trials: ignore the 10 forced heads, just observe the next flip.
n_trials = 100_000
eleventh_flips = np.random.randint(0, 2, size=n_trials)  # 0=tails, 1=heads

p_heads_after_10 = eleventh_flips.mean()
print(f"After 10 heads in a row, P(heads on next flip) ≈ {p_heads_after_10:.4f}")
print(f"Expected: 0.5000")
print(f"The coin has no memory.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Show that a streak of heads doesn't cause compensating tails —
# it just becomes a smaller fraction of a larger total.

np.random.seed(99)

# 10 forced heads, then 990 fair flips
forced  = np.ones(10)                        # 10 heads
fair    = np.random.randint(0, 2, size=990)  # 990 fair flips
flips   = np.concatenate([forced, fair])

# Running proportion of heads
cumulative_heads = np.cumsum(flips)
running_prop     = cumulative_heads / np.arange(1, len(flips) + 1)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(running_prop, color='steelblue', linewidth=1.5, label='Running proportion of heads')
ax.axhline(0.5, color='red', linestyle='--', linewidth=1.2, label='True probability (0.5)')
ax.axvline(10,  color='orange', linestyle=':', linewidth=1.5, label='End of forced streak')
ax.set_xlabel('Flip number')
ax.set_ylabel('Proportion heads')
ax.set_title('The streak dilutes — it does not get cancelled by compensating tails')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The proportion dips toward 0.5 over time — but notice there is no compensating surge of tails. The streak just becomes a smaller fraction of a larger total. The Law of Large Numbers works by dilution, not by correction.

---

## Exercises

1. A bag contains 5 red marbles and 3 blue marbles. You draw one marble. What is `P(Red OR Blue)`? Does the union formula still work here, and why?
2. Using the card deck, compute `P(Ace OR King)`. Are aces and kings mutually exclusive? How does that simplify the union formula?
3. Roll two dice. Build the full sample space. Compute `P(first die = 3 AND second die = 5)`. Are these independent? Verify by checking whether `P(A) * P(B) == P(A ∩ B)`.
4. A website runs three independent servers, each with a 1% chance of failing on any given day. What is `P(at least one fails)`? Use the complement.

In [ ]:
# Your experiments here


---

## Formal Notation

**Union (OR):**
$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

**Intersection (AND), independent events:**
$$P(A \cap B) = P(A) \cdot P(B)$$

**Complement (NOT):**
$$P(A^c) = 1 - P(A)$$

**Mutual exclusivity:** Events A and B are mutually exclusive if $A \cap B = \emptyset$. Then:
$$P(A \cup B) = P(A) + P(B)$$
(no overlap to subtract)

**Independence:** A and B are independent if:
$$P(A \cap B) = P(A) \cdot P(B)$$

Note: mutual exclusivity and independence are **not** the same thing. Mutually exclusive events (A and B can't both happen) are in fact *dependent* — if A happens, you know B didn't.

---

## Real-World Connection

- **System reliability**: if three servers each have 99% uptime and they fail independently, `P(all up) = 0.99³ ≈ 0.97`. That is the multiplication rule for independent events. Redundancy calculations in infrastructure design rely on exactly this.
- **A/B testing**: when you run two simultaneous tests on different user segments, you are betting that the segments are independent — that being in test A doesn't affect whether a user sees test B. Violations of this assumption cause misleading results.
- **Loot tables and randomised mechanics**: a game with a 1% legendary drop rate and a 5% rare drop rate might let both trigger on the same kill. Whether those are independent events (multiply), mutually exclusive (add), or correlated in some other way is a design decision with real gameplay consequences.
- **Gambler's fallacy in engineering**: random number generators have no memory. After generating the same value five times, the next value is not "less likely" to be the same. Tests that assume deterministic patterns in random sequences are written from the gambler's fallacy.

---

## Summary

- **Union (OR):** `P(A ∪ B) = P(A) + P(B) − P(A ∩ B)` — subtract the overlap to avoid double-counting. In Python: `A | B`.
- **Intersection (AND):** for independent events, `P(A ∩ B) = P(A) × P(B)`. In Python: `A & B`.
- **Complement (NOT):** `P(Aᶜ) = 1 − P(A)`. Often the fastest path to an answer.
- **Independence:** knowing one event occurred tells you nothing about the other. Test: `P(A ∩ B) == P(A) × P(B)`.
- **Gambler's fallacy:** independent events have no memory. A streak has no influence on the next outcome. The Law of Large Numbers works by dilution, not correction.
- Mutual exclusivity and independence are different concepts — don't confuse them.

**Next:** Notebook 10 — Conditional Probability and Bayes' Theorem (what changes when events *are* dependent, and how to update beliefs with evidence).